# ReviewInsight-QA 电商评论洞察问答系统

这个 notebook 做什么：读入 50 条真实风格的电商评论 → 把评论变成向量（让电脑能"理解"语义）→ 实现三个功能：

1. **评论问答**：用大白话提问，AI 根据真实评论回答，并标注引用编号
2. **维度统计**：自动统计大家在音质/续航/佩戴等方面聊得最多、夸和踩各多少
3. **洞察报告**：一键生成"好评点 / 吐槽点 / 购买建议"分析报告

**使用方法**：从上往下，一格一格点运行（每格左上角的小三角），每格运行完再点下一格。

## 第 1 步：粘贴你的 API 密钥

把下面引号里的文字换成你自己的硅基流动 API 密钥（就是上次做医疗 RAG 项目用的那个，原样复制粘贴过来）。

In [ ]:
# ===== 把密钥粘贴到下面这对引号中间，其他什么都不要动 =====
SILICONFLOW_API_KEY = "在这里粘贴你的密钥"

API_BASE = "https://api.siliconflow.cn/v1"
EMBED_MODEL = "BAAI/bge-m3"              # 向量模型（免费）
CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # 对话模型（免费）

if SILICONFLOW_API_KEY == "在这里粘贴你的密钥":
    print("⚠️ 你还没有粘贴密钥！请先粘贴再运行这一格")
else:
    print("✅ 密钥已填写")

## 第 2 步：读入评论数据

读取同目录下的 reviews_sample.csv，并看看数据长什么样。

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os

df = pd.read_csv("reviews_sample.csv")
print(f"共读入 {len(df)} 条评论")
print(f"评分分布：{df['rating'].value_counts().sort_index().to_dict()}")
df.head(10)

## 第 3 步：把所有评论变成向量

调用向量接口，把每条评论转成一串数字（向量），电脑靠它判断"哪条评论和你的问题意思接近"。

第一次运行会调用接口（免费），结果会存成缓存文件；以后重复运行会直接读缓存，不重复调用。

In [ ]:
def get_embeddings(texts):
    """调用硅基流动向量接口，把一批文字变成向量"""
    r = requests.post(
        f"{API_BASE}/embeddings",
        headers={"Authorization": f"Bearer {SILICONFLOW_API_KEY}"},
        json={"model": EMBED_MODEL, "input": texts},
        timeout=60,
    )
    r.raise_for_status()
    data = sorted(r.json()["data"], key=lambda x: x["index"])
    return np.array([d["embedding"] for d in data])

CACHE = "review_vectors.npy"
if os.path.exists(CACHE):
    vectors = np.load(CACHE)
    print("✅ 找到缓存的向量，直接使用，不再调用接口")
else:
    texts = df["content"].tolist()
    vecs = []
    for i in range(0, len(texts), 20):  # 每 20 条一批
        vecs.append(get_embeddings(texts[i:i+20]))
        print(f"向量化进度：{min(i+20, len(texts))}/{len(texts)}")
        time.sleep(1)
    vectors = np.vstack(vecs)
    np.save(CACHE, vectors)
    print("✅ 全部评论已向量化并存入缓存")

print("向量形状：", vectors.shape)

## 第 4 步：功能一 —— 评论问答（带引用）

定义问答功能：你的问题 → 找到意思最接近的 5 条评论 → AI 只根据这 5 条评论回答，并标注引用了哪条。

In [ ]:
def retrieve(question, top_k=5):
    """找到和问题意思最接近的 top_k 条评论"""
    qv = get_embeddings([question])[0]
    sims = vectors @ qv / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(qv))
    top = sims.argsort()[::-1][:top_k]
    return df.iloc[top], sims[top]

def ask(question, top_k=5):
    """根据真实评论回答问题，附引用"""
    hits, sims = retrieve(question, top_k)
    context = "\n".join(
        f"[评论{row.review_id}]（{row.rating}星）{row.content}"
        for _, row in hits.iterrows()
    )
    prompt = (
        "你是电商评论分析助手。请只根据下面的真实用户评论回答问题，"
        "每句话末尾标注引用编号（如[评论3]）。评论里没有的信息就直说没有。\n\n"
        f"{context}\n\n问题：{question}"
    )
    r = requests.post(
        f"{API_BASE}/chat/completions",
        headers={"Authorization": f"Bearer {SILICONFLOW_API_KEY}"},
        json={"model": CHAT_MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.3},
        timeout=120,
    )
    r.raise_for_status()
    answer = r.json()["choices"][0]["message"]["content"]
    print("❓ 问题：", question)
    print("=" * 55)
    print("💬 回答：", answer)
    print("=" * 55)
    print("📎 参考的原始评论：")
    for (_, row), s in zip(hits.iterrows(), sims):
        print(f"  [评论{row.review_id}]（{row.rating}星，相似度{s:.2f}）{row.content}")

print("✅ 问答功能已就绪")

## 第 5 步：试试问三个问题

运行这一格，看看 AI 怎么基于真实评论回答。

In [ ]:
ask("这款耳机的续航到底怎么样？")
print()
ask("大家吐槽最多的问题是什么？")
print()
ask("学生党值得买吗？")

## 第 6 步：功能二 —— 维度统计

自动统计每个维度（音质/续航/佩戴舒适度等）被提及多少次、好评差评各多少。

In [ ]:
DIMENSIONS = {
    "音质":     ["音质", "声音", "低音", "高音", "听歌", "调音"],
    "续航":     ["续航", "电量", "充电", "电池"],
    "佩戴舒适度": ["戴着", "佩戴", "耳朵", "舒服", "夹耳", "胀痛", "耳帽"],
    "连接稳定性": ["连接", "蓝牙", "断连", "延迟", "卡顿"],
    "物流包装":  ["物流", "快递", "发货", "包装", "开箱"],
    "客服售后":  ["客服", "售后", "退货", "换货", "维修"],
    "降噪":     ["降噪", "隔音", "噪音"],
    "性价比":    ["性价比", "价位", "便宜", "价格", "值"],
}

rows = []
for dim, kws in DIMENSIONS.items():
    mask = df["content"].apply(lambda t: any(k in t for k in kws))
    sub = df[mask]
    rows.append((dim, len(sub), int((sub["rating"] >= 4).sum()), int((sub["rating"] <= 2).sum())))

stat = pd.DataFrame(rows, columns=["维度", "提及次数", "好评数", "差评数"]).sort_values("提及次数", ascending=False)
print(stat.to_string(index=False))
print()
print("—— 提及热度条形图 ——")
for _, r in stat.iterrows():
    print(f"{r['维度']}\t{'█' * int(r['提及次数'])} {int(r['提及次数'])}")

## 第 7 步：功能三 —— 一键生成洞察报告

把好评和差评原文交给 AI，自动生成一份可以直接用在运营/市场工作里的分析报告。

In [ ]:
pos_text = "\n".join(df[df.rating >= 4]["content"].head(25))
neg_text = "\n".join(df[df.rating <= 2]["content"].head(25))

report_prompt = f"""你是电商运营分析师。根据以下真实用户评论，输出一份产品洞察报告，包含三部分：
一、核心好评点（最多3条，每条一句话，说明是哪个卖点）
二、集中吐槽点（最多3条，每条一句话，并给出一条改进建议）
三、给潜在买家的购买建议（适合什么人买、不适合什么人买）

【好评原文】
{pos_text}

【差评原文】
{neg_text}"""

r = requests.post(
    f"{API_BASE}/chat/completions",
    headers={"Authorization": f"Bearer {SILICONFLOW_API_KEY}"},
    json={"model": CHAT_MODEL, "messages": [{"role": "user", "content": report_prompt}], "temperature": 0.5},
    timeout=120,
)
r.raise_for_status()
print(r.json()["choices"][0]["message"]["content"])

## 跑完了！

到这里三个功能全部跑通。记得**截图保存**：问答结果、维度统计表、洞察报告各截一张，之后做简历和传 GitHub 都要用。